# CogniStream — Data Cleaning and Validation

## Objective
Prepare the AI Developer Productivity dataset for analysis.

## Cleaning process
1. Validate the schema.
2. Convert all feature columns to numeric types.
3. Validate counts, hours, cognitive load, and task success.
4. Preserve legitimate outliers and record invalid values.
5. Save the processed dataset and cleaning report.

## Data integrity
The original CSV in dataset/raw/ remains unchanged.

In [10]:
from pathlib import Path
import sys

current_dir = Path.cwd().resolve()

candidates = [
    current_dir,
    *current_dir.parents,
]

PROJECT_ROOT = next(
    (
        folder
        for folder in candidates
        if (folder / "src" / "schema.py").is_file()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Cannot locate CogniStream project root. "
        "Open the correct project folder in VS Code."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: C:\PlayGround\Infotact Data Analytics\CogniStream


In [11]:
import pandas as pd
import plotly.express as px
from IPython.display import display

from src.data_collection import (
    load_dataset,
    validate_schema,
)

from src.cleaning import (
    clean_dataset,
    build_cleaning_report,
    PROCESSED_DATA_PATH,
    REPORT_PATH,
)

print("Cleaning notebook initialized successfully.")

Cleaning notebook initialized successfully.


In [12]:
RAW_DATA_PATH = (
    PROJECT_ROOT
    / "dataset"
    / "raw"
    / "ai_dev_productivity.csv"
)

raw_df = load_dataset(RAW_DATA_PATH)

# Keep a reference copy for the integrity check.
raw_snapshot = raw_df.copy(deep=True)

print("Raw dataset shape:", raw_df.shape)
display(raw_df.head())

Raw dataset shape: (500, 9)


,hours_coding,coffee_intake_mg,distractions,sleep_hours,commits,bugs_reported,ai_usage_hours,cognitive_load,task_success
0,5.99,600,1,5.8,2,1,0.71,5.4,1
1,4.72,568,2,6.9,5,3,1.75,4.7,1
2,6.30,560,1,8.9,2,0,2.27,2.2,1
3,8.05,600,7,6.3,9,5,1.40,5.9,0
4,4.53,421,6,6.9,4,0,1.26,6.3,1


In [13]:
schema_report = validate_schema(raw_df)

if not schema_report["schema_valid"]:
    raise ValueError(
        f"Invalid schema: {schema_report['missing_columns']}"
    )

cleaned_df = clean_dataset(raw_df)

print("Original shape:", raw_df.shape)
print("Cleaned shape:", cleaned_df.shape)

display(cleaned_df.head())

Original shape: (500, 9)
Cleaned shape: (500, 9)


,hours_coding,coffee_intake_mg,distractions,sleep_hours,commits,bugs_reported,ai_usage_hours,cognitive_load,task_success
0,5.99,600.0,1.0,5.8,2.0,1.0,0.71,5.4,1.0
1,4.72,568.0,2.0,6.9,5.0,3.0,1.75,4.7,1.0
2,6.30,560.0,1.0,8.9,2.0,0.0,2.27,2.2,1.0
3,8.05,600.0,7.0,6.3,9.0,5.0,1.40,5.9,0.0
4,4.53,421.0,6.0,6.9,4.0,0.0,1.26,6.3,1.0


In [14]:
cleaning_report = build_cleaning_report(
    raw_df,
    cleaned_df,
)

display(cleaning_report)

,column,original_missing,newly_invalid,final_missing
0,hours_coding,0,0,0
1,coffee_intake_mg,0,0,0
2,distractions,0,0,0
3,sleep_hours,0,0,0
4,commits,0,0,0
5,bugs_reported,0,0,0
6,ai_usage_hours,0,0,0
7,cognitive_load,0,0,0
8,task_success,0,0,0


In [15]:
validation_results = {
    "Expected columns present": (
        list(cleaned_df.columns)
        == list(raw_df.columns)
    ),
    "No rows lost": (
        len(cleaned_df) == len(raw_df)
    ),
    "No new missing values": (
        int(cleaned_df.isna().sum().sum())
        == int(raw_df.isna().sum().sum())
    ),
    "Raw DataFrame unchanged": (
        raw_df.equals(raw_snapshot)
    ),
    "Task success contains only 0 or 1": (
        cleaned_df["task_success"]
        .dropna()
        .isin([0, 1])
        .all()
    ),
    "Cognitive load stays within 1–10": (
        cleaned_df["cognitive_load"]
        .dropna()
        .between(1, 10)
        .all()
    ),
}

validation_df = pd.DataFrame(
    validation_results.items(),
    columns=["Validation", "Passed"],
)

display(validation_df)

if not validation_df["Passed"].all():
    raise ValueError(
        "One or more cleaning validation checks failed."
    )

,Validation,Passed
0,Expected columns present,True
1,No rows lost,True
2,No new missing values,True
3,Raw DataFrame unchanged,True
4,Task success contains only 0 or 1,True
5,Cognitive load stays within 1–10,True


In [16]:
# comparison_df = pd.DataFrame({
#     "Before cleaning": pd.to_numeric(
#         raw_df["cognitive_load"],
#         errors="coerce",
#     ),
#     "After cleaning": cleaned_df["cognitive_load"],
# })

# comparison_df = comparison_df.melt(
#     var_name="Dataset",
#     value_name="Cognitive load",
# )

# fig = px.histogram(
#     comparison_df,
#     x="Cognitive load",
#     color="Dataset",
#     barmode="overlay",
#     opacity=0.65,
#     nbins=20,
#     title="Cognitive Load Distribution: Before vs After Cleaning",
# )

# fig.update_layout(
#     template="plotly_white",
#     xaxis_title="Cognitive load (1–10)",
#     yaxis_title="Number of observations",
# )

# fig.show()

In [17]:
PROCESSED_DATA_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

cleaned_df.to_csv(
    PROCESSED_DATA_PATH,
    index=False,
)

REPORT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

cleaning_report.to_csv(
    REPORT_PATH,
    index=False,
)

print("Cleaned dataset saved:")
print(PROCESSED_DATA_PATH)

print("\nCleaning report saved:")
print(REPORT_PATH)

Cleaned dataset saved:
C:\PlayGround\Infotact Data Analytics\CogniStream\dataset\processed\cleaned_data.csv

Cleaning report saved:
C:\PlayGround\Infotact Data Analytics\CogniStream\reports\cleaning_report.csv


In [18]:
print("=" * 55)
print("COGNISTREAM — CLEANING COMPLETED")
print("=" * 55)

print(f"Original records: {len(raw_df)}")
print(f"Processed records: {len(cleaned_df)}")
print(f"Original features: {len(raw_df.columns)}")
print(f"Processed features: {len(cleaned_df.columns)}")

print(
    "Total invalid values introduced as missing:",
    int(
        (
            raw_df.notna()
            & cleaned_df.isna()
        ).sum().sum()
    ),
)

print(
    "Raw dataset remains unchanged:",
    raw_df.equals(raw_snapshot),
)

COGNISTREAM — CLEANING COMPLETED
Original records: 500
Processed records: 500
Original features: 9
Processed features: 9
Total invalid values introduced as missing: 0
Raw dataset remains unchanged: True
